In [ ]:
# Detecting Uncontrolled Anger in Social Media Using NLP

#This notebook explores uncontrolled anger detection in social media text from Reddit and Twitter using both traditional machine learning models and transformer-based deep learning.

## Models used
#- Logistic Regression
#- Support Vector Machine (SVM)
#- DistilBERT

## Datasets
#- GoEmotions (Reddit)
#- SemEval-2018 Twitter E-c
#- SemEval-2018 Twitter EI-reg

In [ ]:
########################################
#         IMPORT DATASET
########################################
import pandas as pd

# ---- Read exactly the files you uploaded in the current Colab working dir ----
geo1 = pd.read_csv("goemotions_1.csv")  # has: id, text, ... + 28 emotion columns + neutral (+ example_very_unclear)
geo2 = pd.read_csv("goemotions_2.csv")
geo3 = pd.read_csv("goemotions_3.csv")

# Quick sanity peek (no heavy ops)
print(geo1.shape, geo1.columns[:12].tolist())
print(geo2.shape, geo2.columns[:12].tolist())
print(geo3.shape, geo3.columns[:12].tolist())


In [ ]:
###########################################
#            LOAD DATASET
###########################################


import pandas as pd

# ---- E-c: multi-label emotions (11 labels) ----
ec_train = pd.read_csv("2018-E-c-En-train.txt", sep="\t", encoding="utf-8")
ec_dev   = pd.read_csv("2018-E-c-En-dev.txt",   sep="\t", encoding="utf-8")
ec_test  = pd.read_csv("2018-E-c-En-test-gold.txt", sep="\t", encoding="utf-8")

# Normalize tweet column name → 'text' (optional, for later consistency)
if "Tweet" in ec_train.columns: ec_train = ec_train.rename(columns={"Tweet": "text"})
if "Tweet" in ec_dev.columns:   ec_dev   = ec_dev.rename(columns={"Tweet": "text"})
if "Tweet" in ec_test.columns:  ec_test  = ec_test.rename(columns={"Tweet": "text"})

print(ec_train.shape, ec_train.columns[:12].tolist())
print(ec_dev.shape,   ec_dev.columns[:12].tolist())
print(ec_test.shape,  ec_test.columns[:12].tolist())


In [ ]:
import pandas as pd

# ---- EI-reg: anger intensity (continuous 0..1) ----
reg_train = pd.read_csv("EI-reg-En-anger-train.txt", sep="\t", encoding="utf-8")
reg_dev   = pd.read_csv("2018-EI-reg-En-anger-dev.txt",   sep="\t", encoding="utf-8")
reg_test  = pd.read_csv("2018-EI-reg-En-anger-test-gold.txt", sep="\t", encoding="utf-8")

# Normalize tweet column name → 'text' (optional)
for df in (reg_train, reg_dev, reg_test):
    if "Tweet" in df.columns:
        df.rename(columns={"Tweet": "text"}, inplace=True)

# Try to standardize intensity column name to 'anger_intensity' (name varies across bundles)
def standardize_intensity(df):
    for c in df.columns:
        if c.lower() in {"intensity","score","anger-intensity","gold"}:
            df.rename(columns={c: "anger_intensity"}, inplace=True)
            return
    # fallback: if 3+ cols, last one is usually the score
    if df.shape[1] >= 3:
        df.rename(columns={df.columns[-1]: "anger_intensity"}, inplace=True)

standardize_intensity(reg_train)
standardize_intensity(reg_dev)
standardize_intensity(reg_test)

print(reg_train.shape, reg_train.columns.tolist())
print(reg_dev.shape,   reg_dev.columns.tolist())
print(reg_test.shape,  reg_test.columns.tolist())


In [ ]:
##########################################
#     Data Cleaning and Preprocessing
##########################################

# GoEmotions (CSV) -> single clean CSV
import pandas as pd

# Read the exact uploaded files in current working dir
geo1 = pd.read_csv("goemotions_1.csv")
geo2 = pd.read_csv("goemotions_2.csv")
geo3 = pd.read_csv("goemotions_3.csv")

EMOTION_COLS = [
    "admiration","amusement","anger","annoyance","approval","caring","confusion","curiosity",
    "desire","disappointment","disapproval","disgust","embarrassment","excitement","fear",
    "gratitude","grief","joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]

# Keep only needed columns that actually exist
def keep_cols(df):
    cols = ["id","text","example_very_unclear"] + [c for c in EMOTION_COLS if c in df.columns]
    return df[cols]

geo = pd.concat([keep_cols(geo1), keep_cols(geo2), keep_cols(geo3)], ignore_index=True)

# Drop ambiguous if present
if "example_very_unclear" in geo.columns:
    geo = geo[geo["example_very_unclear"].fillna(0) == 0]
    geo = geo.drop(columns=["example_very_unclear"], errors="ignore")

# Deduplicate by Reddit comment id
geo = geo.drop_duplicates(subset="id")

# Save
geo.to_csv("reddit_goemotions.csv", index=False)
print("Saved reddit_goemotions.csv ->", geo.shape)


In [ ]:
# SemEval E-c (English) -> CSVs with unified column names
import pandas as pd

ec_train = pd.read_csv("2018-E-c-En-train.txt", sep="\t", encoding="utf-8")
ec_dev   = pd.read_csv("2018-E-c-En-dev.txt",   sep="\t", encoding="utf-8")
ec_test  = pd.read_csv("2018-E-c-En-test-gold.txt", sep="\t", encoding="utf-8")

for df in (ec_train, ec_dev, ec_test):
    if "Tweet" in df.columns:
        df.rename(columns={"Tweet": "text"}, inplace=True)
    if "ID" in df.columns:
        df.rename(columns={"ID": "id"}, inplace=True)

EC_LABELS = ["anger","anticipation","disgust","fear","joy","love","optimism","pessimism","sadness","surprise","trust"]

ec_train[["id","text"] + EC_LABELS].to_csv("twitter_ec_train.csv", index=False)
ec_dev[  ["id","text"] + EC_LABELS].to_csv("twitter_ec_dev.csv",   index=False)
ec_test[ ["id","text"] + EC_LABELS].to_csv("twitter_ec_test.csv",  index=False)

print("Saved twitter_ec_{train,dev,test}.csv")


In [ ]:
# SemEval EI-reg (anger) -> CSVs with standardized anger_intensity
import pandas as pd

reg_train = pd.read_csv("EI-reg-En-anger-train.txt", sep="\t", encoding="utf-8")
reg_dev   = pd.read_csv("2018-EI-reg-En-anger-dev.txt",   sep="\t", encoding="utf-8")
reg_test  = pd.read_csv("2018-EI-reg-En-anger-test-gold.txt", sep="\t", encoding="utf-8")

for df in (reg_train, reg_dev, reg_test):
    # unify names
    if "Tweet" in df.columns: df.rename(columns={"Tweet":"text"}, inplace=True)
    if "ID" in df.columns:    df.rename(columns={"ID":"id"}, inplace=True)
    # standardize intensity column name
    int_col = None
    for c in df.columns:
        if c.lower() in {"anger_intensity","intensity","score","gold","anger-intensity"}:
            int_col = c; break
    if int_col is None and df.shape[1] >= 3:
        int_col = df.columns[-1]
    df.rename(columns={int_col: "anger_intensity"}, inplace=True)
    # drop constant meta column if present
    if "Affect Dimension" in df.columns:
        df.drop(columns=["Affect Dimension"], inplace=True)

reg_train[["id","text","anger_intensity"]].to_csv("twitter_eireg_anger_train.csv", index=False)
reg_dev[  ["id","text","anger_intensity"]].to_csv("twitter_eireg_anger_dev.csv",   index=False)
reg_test[ ["id","text","anger_intensity"]].to_csv("twitter_eireg_anger_test.csv",  index=False)

print("Saved twitter_eireg_anger_{train,dev,test}.csv")


In [ ]:
import pandas as pd

# --- Reddit ---
reddit = pd.read_csv("reddit_goemotions.csv")

# --- Twitter E-c ---
ec_train = pd.read_csv("twitter_ec_train.csv")
ec_dev   = pd.read_csv("twitter_ec_dev.csv")
ec_test  = pd.read_csv("twitter_ec_test.csv")

# --- Twitter EI-reg (anger) ---
reg_train = pd.read_csv("twitter_eireg_anger_train.csv")
reg_dev   = pd.read_csv("twitter_eireg_anger_dev.csv")
reg_test  = pd.read_csv("twitter_eireg_anger_test.csv")

# Quick check of shapes
print("Reddit:", reddit.shape)
print("Twitter E-c:", ec_train.shape, ec_dev.shape, ec_test.shape)
print("Twitter EI-reg:", reg_train.shape, reg_dev.shape, reg_test.shape)


## Operationalising Uncontrolled Anger

Since the datasets do not explicitly label uncontrolled anger, it is operationalised using dataset-specific rules.

For Reddit (GoEmotions), uncontrolled anger is defined as comments labelled with **anger** or **annoyance**.

For Twitter EI-reg, uncontrolled anger is defined using **anger intensity thresholds** (e.g., values above 0.5).

In [ ]:
###############################################
#      Operationalising Uncontrolled Anger
###############################################


# --- Reddit GoEmotion---
# labeled_uncotrolled = 1 if anger==1 or annoyance==1
reddit['labelled_uncontrolled'] = ((reddit['anger'] ==1) | (reddit['annoyance'] ==1)).astype(int)

print("Reddit uncontrolled anger distribution:")
print(reddit['labelled_uncontrolled'].value_counts())

#---Twitter E-c ---
# labelled_unctrolled = 1 if anger==1
ec_train['label_uncontrolled'] = ec_train['anger']
ec_dev['label_uncontrolled'] = ec_dev['anger']
ec_test['label_uncontrolled'] = ec_test['anger']

print("\nTwitter E-c uncontrolled anger distribution:")
print("Train:", ec_train['label_uncontrolled'].value_counts().to_dict())
print("Dev:", ec_dev['label_uncontrolled'].value_counts().to_dict())
print("Test:", ec_test['label_uncontrolled'].value_counts().to_dict())

# --- Twitter EI-reg ---
# Create binary lables for thersholds 0.5, 0.6, 0.7
for thr in [0.5, 0.6, 0.7]:
  col_name = f'label_unctrolled_{str(thr).replace(".", "")}'
  reg_train[col_name] = (reg_train['anger_intensity'] > thr). astype(int)
  reg_dev[col_name] = (reg_dev['anger_intensity'] > thr). astype(int)
  reg_test[col_name] = (reg_test['anger_intensity'] > thr). astype(int)

print("\nTwitter EI-reg uncontrolled anger distribution (train, thr=0.5/0.6/0.7):")
for col in [c for c in reg_train.columns if c.startswith("label_uncontrolled_")]:
    print(col, reg_train[col].value_counts().to_dict())

In [ ]:
# Check distributions for EI-reg with thresholds
thresholds = [0.5, 0.6, 0.7]

for thr in thresholds:
    col_name = f"label_uncontrolled_{str(thr).replace('.', '')}"
    reg_train[col_name] = (reg_train['anger_intensity'] > thr).astype(int)
    reg_dev[col_name]   = (reg_dev['anger_intensity'] > thr).astype(int)
    reg_test[col_name]  = (reg_test['anger_intensity'] > thr).astype(int)

    print(f"\nThreshold = {thr}")
    print("Train:", reg_train[col_name].value_counts().to_dict())
    print("Dev:",   reg_dev[col_name].value_counts().to_dict())
    print("Test:",  reg_test[col_name].value_counts().to_dict())


In [ ]:
# --- Reddit ---
reddit['label_uncontrolled'] = ((reddit['anger'] == 1) | (reddit['annoyance'] == 1)).astype(int)

# --- Twitter E-c ---
ec_train['label_uncontrolled'] = ec_train['anger']
ec_dev['label_uncontrolled']   = ec_dev['anger']
ec_test['label_uncontrolled']  = ec_test['anger']

# --- Twitter EI-reg ---
# (label_uncontrolled_05/06/07)
cols_to_drop = [c for c in reg_train.columns if c.startswith("label_unctrolled")]
reg_train.drop(columns=cols_to_drop, inplace=True, errors="ignore")
reg_dev.drop(columns=cols_to_drop, inplace=True, errors="ignore")
reg_test.drop(columns=cols_to_drop, inplace=True, errors="ignore")

print ("OK")


In [ ]:

#########################################
#   Baseline Model
#########################################

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline

# Helper function for training + evaluation
def run_baseline(X_train, y_train, X_test, y_test, model_type="logreg"):
    if model_type == "logreg":
        model = LogisticRegression(max_iter=200, class_weight="balanced")
    elif model_type == "svm":
        model = LinearSVC(class_weight="balanced")

    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
        ('clf', model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    results = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

    # ROC-AUC only for Logistic Regression
    if model_type == "logreg":
        y_prob = pipe.predict_proba(X_test)[:, 1]
        results["AUC"] = roc_auc_score(y_test, y_prob)

    return results


# --- Reddit ---
X_train, y_train = reddit["text"], reddit["label_uncontrolled"]
reddit_results_logreg = run_baseline(X_train, y_train, X_train, y_train, "logreg")
reddit_results_svm = run_baseline(X_train, y_train, X_train, y_train, "svm")

print("\nReddit Results (LogReg):", reddit_results_logreg)
print("Reddit Results (SVM):", reddit_results_svm)


# --- Twitter E-c (train vs test split) ---
X_train, y_train = ec_train["text"], ec_train["label_uncontrolled"]
X_test, y_test   = ec_test["text"], ec_test["label_uncontrolled"]

ec_results_logreg = run_baseline(X_train, y_train, X_test, y_test, "logreg")
ec_results_svm    = run_baseline(X_train, y_train, X_test, y_test, "svm")

print("\nTwitter E-c Results (LogReg):", ec_results_logreg)
print("Twitter E-c Results (SVM):", ec_results_svm)


# --- Twitter EI-reg (threshold=0.5, train vs test) ---
X_train, y_train = reg_train["text"], reg_train["label_uncontrolled_05"]
X_test, y_test   = reg_test["text"], reg_test["label_uncontrolled_05"]

reg_results_logreg = run_baseline(X_train, y_train, X_test, y_test, "logreg")
reg_results_svm    = run_baseline(X_train, y_train, X_test, y_test, "svm")

print("\nTwitter EI-reg (thr=0.5) Results (LogReg):", reg_results_logreg)
print("Twitter EI-reg (thr=0.5) Results (SVM):", reg_results_svm)


beacuse Reddit has no test  data we choose the train data for baseline and and it might the overfit raise so cross validation should be check

In [ ]:
###############################################
#       Cross-Validation
###############################################


from sklearn.model_selection import cross_validate, StratifiedKFold
def run_cv(X, y, model_type='logreg'):
  if model_type == "logreg":
    model = LogisticRegression(max_iter = 200, class_weight = "balanced")
  elif model_type == "svm":
    model = LinearSVC(class_weight="balanced")

  pipe = Pipeline([
      ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
      ('clf', model)
  ])
# Stratified 5-fold CV
  cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1'
  }

  results = cross_validate(pipe, X, y, cv=cv, scoring=scoring, return_train_score=False)
  return {metric: results[f'test_{metric}'].mean() for metric in scoring.keys()}

#--- Run CV on Reddit ---
X, y = reddit["text"], reddit["label_uncontrolled"]

reddit_cv_logreg = run_cv(X,y, "logreg")
reddit_cv_svm = run_cv(X,y, "svm")

print("Reddit 5-Fold CV Results (LogReg):", reddit_cv_logreg)
print("Reddit 5-Fold CV Results (SVM):", reddit_cv_svm)


this analyse shows that the previous model is too sympatich and we use more developed model than transformer

In [ ]:
########################################
#       Cross-Domain Evaluation
########################################


# --- Helper: run baseline cross-domain ---
def run_cross_domain(X_train, y_train, X_test, y_test, model_type="logreg"):
  if model_type == "logreg":
    model = LogisticRegression(max_iter=200, class_weight="balanced")
  elif model_type == "svm":
    model = LinearSVC(class_weight="balanced")

  pipe = Pipeline([
      ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
      ('clf', model)
  ])

  pipe.fit(X_train, y_train)
  y_pred = pipe.predict(X_test)

  results = {
      "Accuracy": accuracy_score(y_test, y_pred),
      "Precision": precision_score(y_test, y_pred),
      "Recall": recall_score(y_test, y_pred),
      "F1": f1_score(y_test, y_pred)
  }

  # AUC only for LegReg
  if model_type == "logreg":
      try:
          y_prob = pipe.predict_proba(X_test)[:,1]
          results["AUC"] = roc_auc_score(y_test, y_prob)
      except:
          pass

  return results
# --- Cross-domain: Twitter (E-c Train) to Reddit ---
X_train, y_train = ec_train["text"], ec_train["label_uncontrolled"]
X_test, y_test   = reddit["text"], reddit["label_uncontrolled"]

cross_twitter_to_reddit_logreg = run_cross_domain(X_train, y_train, X_test, y_test, "logreg")
cross_twitter_to_reddit_svm    = run_cross_domain(X_train, y_train, X_test, y_test, "svm")

print("\nTwitter -> Reddit (LogReg):", cross_twitter_to_reddit_logreg)
print("Twitter -> Reddit (SVM):", cross_twitter_to_reddit_svm)


# --- Cross-domain: Reddit -> Twitter (E-c test) ---
X_train, y_train = reddit["text"], reddit["label_uncontrolled"]
X_test, y_test   = ec_test["text"], ec_test["label_uncontrolled"]

cross_reddit_to_twitter_logreg = run_cross_domain(X_train, y_train, X_test, y_test, "logreg")
cross_reddit_to_twitter_svm    = run_cross_domain(X_train, y_train, X_test, y_test, "svm")

print("\nReddit -> Twitter (LogReg):", cross_reddit_to_twitter_logreg)
print("Reddit -> Twitter (SVM):", cross_reddit_to_twitter_svm)


This section fine-tunes DistilBERT to compare contextual transformer performance against traditional machine learning baselines.

In [ ]:
######################################
#     DistilBERT Experiments
######################################


!pip install transformers torch --quiet

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Step 1: Dataset class ---
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_len)
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# --- Step 2: Evaluation metrics ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

# --- Step 3: Training function ---
def run_distilbert(train_texts, train_labels, test_texts, test_labels, epochs=2):
    import os
    os.environ["WANDB_DISABLED"] = "true"  # disable wandb logs

    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

    train_dataset = TextDataset(train_texts, train_labels, tokenizer)
    test_dataset = TextDataset(test_texts, test_labels, tokenizer)

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        logging_dir="./logs",
        logging_steps=50,
        save_strategy="no"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    return results


In [ ]:
ec_results_distilbert = run_distilbert(
    ec_train["text"], ec_train["label_uncontrolled"],
    ec_test["text"], ec_test["label_uncontrolled"],
    epochs=2
)
print("Twitter E-c (DistilBERT):", ec_results_distilbert)



In [ ]:
ei_results_distilbert = run_distilbert(
    reg_train["text"], reg_train["label_uncontrolled_05"],
    reg_test["text"], reg_test["label_uncontrolled_05"],
    epochs=2
)

print("Twitter EI-reg (DistilBERT):", ei_results_distilbert)


In [ ]:
from sklearn.model_selection import train_test_split

# --- Split Reddit 80/20 ---
X_train, X_test, y_train, y_test = train_test_split(
    reddit["text"], reddit["label_uncontrolled"], test_size=0.2, random_state=42, stratify=reddit["label_uncontrolled"]
)

# --- Run DistilBERT ---
reddit_results_distilbert = run_distilbert(
    X_train, y_train,
    X_test, y_test,
    epochs=2
)

print("Reddit (DistilBERT):", reddit_results_distilbert)


In [ ]:
# --- Cross-domain ---
# --- Twitter (E-c train) -> Reddit (test) ---
X_train, y_train = ec_train["text"], ec_train["label_uncontrolled"]
X_test, y_test   = reddit["text"], reddit["label_uncontrolled"]

cross_twitter_to_reddit = run_distilbert(X_train, y_train, X_test, y_test, epochs=2)
print("DistilBERT Twitter -> Reddit:", cross_twitter_to_reddit)


# --- Reddit (train 80%) -> Twitter (E-c test) ---
from sklearn.model_selection import train_test_split

X_rtrain, X_rtest, y_rtrain, y_rtest = train_test_split(
    reddit["text"], reddit["label_uncontrolled"], test_size=0.2, random_state=42, stratify=reddit["label_uncontrolled"]
)

cross_reddit_to_twitter = run_distilbert(X_rtrain, y_rtrain, ec_test["text"], ec_test["label_uncontrolled"], epochs=2)
print("DistilBERT Reddit -> Twitter:", cross_reddit_to_twitter)


In [ ]:
########################################
#       Results Summary
########################################

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np

# --- Train on Twitter E-c (train split) ---
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(ec_train["text"])
y_train = ec_train["label_uncontrolled"]

model = LogisticRegression(max_iter=200, class_weight="balanced")
model.fit(X_train, y_train)

# --- Get feature importance ---
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = model.coef_[0]

# Top 20 strongest indicators of anger (positive coefficients)
top_pos_idx = np.argsort(coefs)[-20:]
top_pos_features = feature_names[top_pos_idx]
top_pos_weights = coefs[top_pos_idx]

# Top 20 strongest indicators of non-anger (negative coefficients)
top_neg_idx = np.argsort(coefs)[:20]
top_neg_features = feature_names[top_neg_idx]
top_neg_weights = coefs[top_neg_idx]

print(" Yes Top indicators of uncontrolled anger:")
for f, w in zip(top_pos_features, top_pos_weights):
    print(f"{f}: {w:.3f}")

print("\n No Top indicators of non-anger:")
for f, w in zip(top_neg_features, top_neg_weights):
    print(f"{f}: {w:.3f}")


In [ ]:
# --- Train on Reddit (GoEmotions) ---
vectorizer_reddit = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_reddit = vectorizer_reddit.fit_transform(reddit["text"])
y_train_reddit = reddit["label_uncontrolled"]

model_reddit = LogisticRegression(max_iter=200, class_weight="balanced")
model_reddit.fit(X_train_reddit, y_train_reddit)

# --- Get feature importance ---
feature_names_reddit = np.array(vectorizer_reddit.get_feature_names_out())
coefs_reddit = model_reddit.coef_[0]

# Top 20 positive features (anger indicators)
top_pos_idx = np.argsort(coefs_reddit)[-20:]
top_pos_features = feature_names_reddit[top_pos_idx]
top_pos_weights = coefs_reddit[top_pos_idx]

# Top 20 negative features (non-anger indicators)
top_neg_idx = np.argsort(coefs_reddit)[:20]
top_neg_features = feature_names_reddit[top_neg_idx]
top_neg_weights = coefs_reddit[top_neg_idx]

print("Yes Top indicators of uncontrolled anger (Reddit):")
for f, w in zip(top_pos_features, top_pos_weights):
    print(f"{f}: {w:.3f}")

print("\n No Top indicators of non-anger (Reddit):")
for f, w in zip(top_neg_features, top_neg_weights):
    print(f"{f}: {w:.3f}")


## Final Notes

- Traditional models provided strong baselines but struggled with cross-domain transfer.
- DistilBERT improved in-domain performance, especially on Twitter datasets.
- Cross-platform generalisation remained challenging.